# Module 2 — Advanced GroupBy & Pivot Tables for Financial Aggregation

> **Track 1 · Data Science Foundations**  
> **Difficulty**: Beginner → Intermediate  
> **Runtime**: ~2 minutes  
> **Key Libraries**: Pandas, NumPy, Matplotlib, Seaborn, Plotly  
> **Prerequisite**: Module 1 (Optimized Pandas)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Data Generation](#3-synthetic-data-generation)
4. [Exploratory Data Analysis](#4-exploratory-data-analysis)
5. [GroupBy Fundamentals](#5-groupby-fundamentals)
6. [Advanced Aggregation with `.agg()`](#6-advanced-aggregation)
7. [Pivot Tables & Multi-Level Indexing](#7-pivot-tables)
8. [Rolling & Expanding Windows](#8-rolling--expanding-windows)
9. [`.transform()` for Feature Engineering](#9-transform-for-feature-engineering)
10. [Visualization Dashboard](#10-visualization-dashboard)
11. [Key Business Takeaway](#11-key-business-takeaway)

---
## §1 · Business Context

### Why do GroupBy and Pivot Tables matter at Revolut?

The Risk & Finance teams at Revolut rely on **aggregated views** of transaction data:

| Team | Question | GroupBy Pattern |
|------|----------|-----------------|
| **Fraud Ops** | "Which users had >5 ATM withdrawals in the last hour?" | Rolling groupby |
| **Finance** | "What's the weekly revenue by currency and merchant category?" | Pivot table |
| **Product** | "How does average spend change over a user's first 30 days?" | Expanding window |
| **Compliance** | "Show me users whose 7-day spend exceeds their historical 3σ threshold" | Transform + rolling |

Mastering **`.groupby()`**, **`.pivot_table()`**, **`.rolling()`**, and **`.transform()`** is essential for turning raw event logs into actionable business metrics.

In this notebook we will:
1. Generate a realistic 1M-row Revolut transaction log.
2. Progress from basic groupby to multi-level pivot tables.
3. Build rolling and expanding window features used in fraud detection.
4. Visualize aggregated metrics for stakeholder dashboards.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", palette="muted")

print("✓ All imports loaded")
print(f"  Pandas {pd.__version__}  |  NumPy {np.__version__}")

---
## §3 · Synthetic Data Generation

We simulate **1 000 000 transactions** across 25 000 users over 180 days,
with realistic patterns: weekday/weekend seasonality, multi-currency, and multiple payment channels.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_ROWS  = 1_000_000
N_USERS = 25_000
START   = pd.Timestamp("2025-01-01")
END     = pd.Timestamp("2025-06-30")
SPAN_S  = (END - START).total_seconds()

CATEGORIES  = ["Grocery", "Travel", "ATM", "Entertainment",
               "Dining", "Subscription", "Shopping", "Utilities"]
CURRENCIES  = ["GBP", "EUR", "USD", "PLN"]
CHANNELS    = ["card", "mobile", "web", "direct_debit"]
COUNTRIES   = ["GB", "DE", "FR", "ES", "IT", "PL", "US", "NL"]

print(f"Generating {N_ROWS:,} transactions for {N_USERS:,} users …")

# User-level base currency (each user has a home currency)
user_currency = pd.Series(
    np.random.choice(["GBP", "EUR", "USD", "PLN"], N_USERS, p=[0.45, 0.30, 0.15, 0.10]),
    index=range(1, N_USERS + 1), name="home_currency"
)

user_ids = np.random.randint(1, N_USERS + 1, size=N_ROWS)

# Timestamps with weekday bias (more txns Mon-Fri)
raw_ts = START + pd.to_timedelta(np.random.uniform(0, SPAN_S, N_ROWS), unit="s")
df = pd.DataFrame({
    "txn_id":            np.arange(1, N_ROWS + 1),
    "user_id":           user_ids,
    "timestamp":         raw_ts,
    "merchant_category": np.random.choice(CATEGORIES, N_ROWS,
                              p=[0.22, 0.06, 0.14, 0.10, 0.22, 0.08, 0.12, 0.06]),
    "currency":          np.random.choice(CURRENCIES, N_ROWS, p=[0.45, 0.28, 0.17, 0.10]),
    "amount":            np.round(np.abs(np.random.lognormal(3.5, 1.2, N_ROWS)), 2),
    "fx_rate":           np.round(np.random.uniform(0.70, 1.40, N_ROWS), 4),
    "channel":           np.random.choice(CHANNELS, N_ROWS, p=[0.40, 0.30, 0.10, 0.20]),
    "country":           np.random.choice(COUNTRIES, N_ROWS, p=[0.35, 0.12, 0.12, 0.10, 0.08, 0.08, 0.10, 0.05]),
})

df.sort_values("timestamp", inplace=True)
df.reset_index(drop=True, inplace=True)

# Add derived time columns (critical for groupby operations)
df["date"]       = df["timestamp"].dt.date
df["year_week"]  = df["timestamp"].dt.isocalendar().week.astype(int)
df["year_month"] = df["timestamp"].dt.to_period("M")
df["day_of_week"]= df["timestamp"].dt.day_name()
df["hour"]       = df["timestamp"].dt.hour
df["is_weekend"] = df["timestamp"].dt.weekday >= 5

# Compute amount_gbp once
df["amount_gbp"] = df["amount"] * df["fx_rate"]

print(f"✓ Shape: {df.shape}")
df.head(10)

In [ ]:
df.info(memory_usage="shallow")

---
## §4 · Exploratory Data Analysis

In [ ]:
# High-level statistics
df[["amount", "amount_gbp", "fx_rate"]].describe()

In [ ]:
# Transactions per user distribution
txns_per_user = df.groupby("user_id").size()

fig = px.histogram(
    txns_per_user, nbins=80,
    title="Distribution of Transactions per User",
    labels={"value": "Transactions per User", "count": "Number of Users"},
    color_discrete_sequence=["#636EFA"],
)
fig.add_vline(txns_per_user.median(), line_dash="dash",
              annotation_text=f"Median: {txns_per_user.median():.0f}")
fig.update_layout(height=400)
fig.show()

In [ ]:
# Daily volume with weekday/weekend coloring
daily = df.groupby("date").agg(
    txn_count=("txn_id", "count"),
    total_gbp=("amount_gbp", "sum"),
).reset_index()
daily["date"] = pd.to_datetime(daily["date"])
daily["is_weekend"] = daily["date"].dt.weekday >= 5

fig = px.bar(daily, x="date", y="txn_count",
             color="is_weekend", color_discrete_map={True: "#EF553B", False: "#00CC96"},
             title="Daily Transaction Volume (green = weekday, red = weekend)")
fig.update_layout(height=380, showlegend=False,
                  xaxis_title="Date", yaxis_title="Transactions")
fig.show()

---
## §5 · GroupBy Fundamentals

### 5.1 — Basic GroupBy: Split → Apply → Combine

The three-step pattern:
1. **Split** the DataFrame by a key column.
2. **Apply** an aggregation function to each group.
3. **Combine** the results into a new DataFrame.

In [ ]:
# Total revenue and transaction count by merchant category
cat_summary = (
    df.groupby("merchant_category")
    .agg(
        txn_count=("txn_id", "count"),
        total_gbp=("amount_gbp", "sum"),
        avg_amount=("amount_gbp", "mean"),
    )
    .sort_values("total_gbp", ascending=False)
)
cat_summary["revenue_share_pct"] = (cat_summary["total_gbp"] / cat_summary["total_gbp"].sum() * 100)
cat_summary

In [ ]:
# Visualize category breakdown
fig = px.treemap(
    cat_summary.reset_index(),
    path=["merchant_category"],
    values="total_gbp",
    title="Revenue Share by Merchant Category",
    color="avg_amount", color_continuous_scale="YlOrRd",
)
fig.update_layout(height=440)
fig.show()

### 5.2 — Multiple Group Keys

In [ ]:
# Revenue by category AND currency
multi_group = (
    df.groupby(["merchant_category", "currency"])
    .agg(
        txn_count=("txn_id", "count"),
        total_gbp=("amount_gbp", "sum"),
    )
    .reset_index()
)
multi_group.sort_values("total_gbp", ascending=False).head(12)

In [ ]:
# Stacked bar: category × currency
fig = px.bar(
    multi_group, x="merchant_category", y="total_gbp",
    color="currency", barmode="stack",
    title="Revenue by Category × Currency (stacked)",
)
fig.update_layout(height=420, xaxis_tickangle=-30,
                  yaxis_title="Total GBP")
fig.show()

### 5.3 — `.filter()` — Keep Entire Groups Matching a Condition

In [ ]:
# Find users who spent more than £10,000 total
user_totals = df.groupby("user_id")["amount_gbp"].sum()
high_spenders = user_totals[user_totals > 10_000].index

print(f"Users with > £10K total spend: {len(high_spenders):,}  "
      f"({len(high_spenders) / N_USERS * 100:.1f}% of all users)")

# .filter() keeps all rows belonging to matching groups
df_high = df.groupby("user_id").filter(lambda g: g["amount_gbp"].sum() > 10_000)
print(f"Rows belonging to high-spenders: {len(df_high):,}")

---
## §6 · Advanced Aggregation with `.agg()`

### 6.1 — Named Aggregation with Multiple Functions

In [ ]:
# Comprehensive stats per channel
channel_stats = (
    df.groupby("channel")["amount_gbp"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        skew="skew",
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
    )
    .round(2)
)
channel_stats

### 6.2 — Custom Aggregation Functions

In [ ]:
# Custom: coefficient of variation and IQR
def cv(x):
    """Coefficient of Variation — measures relative dispersion."""
    return x.std() / x.mean() if x.mean() > 0 else 0

def iqr(x):
    """Interquartile Range."""
    return x.quantile(0.75) - x.quantile(0.25)

custom_agg = (
    df.groupby("merchant_category")["amount_gbp"]
    .agg(["count", "mean", "median", cv, iqr])
    .rename(columns={"cv": "CV", "iqr": "IQR"})
    .round(2)
)
custom_agg

In [ ]:
# Visualize CV — higher CV means more variable spending within that category
fig = px.bar(
    custom_agg.reset_index(),
    x="merchant_category", y="CV",
    color="CV", color_continuous_scale="RdYlGn_r",
    title="Coefficient of Variation by Category (higher = more variable spend)",
)
fig.update_layout(height=380, xaxis_tickangle=-30)
fig.show()

### 6.3 — Aggregation Across Multiple Columns

In [ ]:
# Country-level summary: transactions, revenue, unique users, avg fx_rate
country_summary = (
    df.groupby("country")
    .agg(
        transactions=("txn_id", "count"),
        total_gbp=("amount_gbp", "sum"),
        unique_users=("user_id", "nunique"),
        avg_fx=("fx_rate", "mean"),
    )
    .assign(revenue_per_user=lambda d: d["total_gbp"] / d["unique_users"])
    .sort_values("total_gbp", ascending=False)
    .round(2)
)
country_summary

---
## §7 · Pivot Tables & Multi-Level Indexing

### 7.1 — Basic Pivot Table

In [ ]:
# Revenue pivot: Category (rows) × Currency (columns)
pivot_basic = pd.pivot_table(
    df,
    values="amount_gbp",
    index="merchant_category",
    columns="currency",
    aggfunc="sum",
    fill_value=0,
    margins=True,           # adds row/column totals
    margins_name="Total",
)
pivot_basic.style.format("£{:,.0f}").background_gradient(cmap="YlOrRd", axis=None)

### 7.2 — Multi-Level Pivot: Day of Week × Hour

In [ ]:
# Transaction count: Day of Week × Hour of Day
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

pivot_time = pd.pivot_table(
    df,
    values="txn_id",
    index="day_of_week",
    columns="hour",
    aggfunc="count",
    fill_value=0,
)[day_order]   # reorder rows

fig = px.imshow(
    pivot_time.values,
    x=pivot_time.columns,
    y=pivot_time.index,
    color_continuous_scale="Blues",
    title="Transaction Heatmap: Day of Week × Hour of Day",
    labels=dict(x="Hour", y="Day", color="Transactions"),
    aspect="auto",
)
fig.update_layout(height=420)
fig.show()

### 7.3 — Multi-Function Pivot

In [ ]:
# Multiple aggregations at once: mean AND count
pivot_multi = pd.pivot_table(
    df,
    values="amount_gbp",
    index="merchant_category",
    columns="channel",
    aggfunc=["mean", "count"],
    fill_value=0,
).round(2)
pivot_multi

### 7.4 — `pd.crosstab()` — Quick Frequency Tables

In [ ]:
# Channel × Category frequency (normalized to percentages)
ct = pd.crosstab(df["channel"], df["merchant_category"], normalize="all") * 100
ct.style.format("{:.1f}%").background_gradient(cmap="Blues")

---
## §8 · Rolling & Expanding Windows

### 8.1 — Daily Revenue with Rolling Average

In [ ]:
# Aggregate to daily level first
daily_rev = (
    df.set_index("timestamp")
    .resample("D")["amount_gbp"]
    .sum()
    .to_frame("daily_revenue")
)

# Rolling windows
daily_rev["MA_7d"]  = daily_rev["daily_revenue"].rolling(7, min_periods=1).mean()
daily_rev["MA_30d"] = daily_rev["daily_revenue"].rolling(30, min_periods=1).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_rev.index, y=daily_rev["daily_revenue"],
                         mode="lines", name="Daily Revenue",
                         line=dict(color="lightgray", width=1)))
fig.add_trace(go.Scatter(x=daily_rev.index, y=daily_rev["MA_7d"],
                         mode="lines", name="7-day MA",
                         line=dict(color="#636EFA", width=2.5)))
fig.add_trace(go.Scatter(x=daily_rev.index, y=daily_rev["MA_30d"],
                         mode="lines", name="30-day MA",
                         line=dict(color="#EF553B", width=2.5)))
fig.update_layout(title="Daily Revenue with Moving Averages",
                  yaxis_title="Revenue (GBP)", height=420,
                  legend=dict(orientation="h", y=1.12))
fig.show()

### 8.2 — Expanding Window: Cumulative Metrics

In [ ]:
# Expanding sum = cumulative revenue YTD
daily_rev["cum_revenue"]  = daily_rev["daily_revenue"].expanding().sum()
daily_rev["cum_avg"]      = daily_rev["daily_revenue"].expanding().mean()

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("Cumulative Revenue (YTD)", "Running Average Revenue"))

fig.add_trace(go.Scatter(x=daily_rev.index, y=daily_rev["cum_revenue"],
                         fill="tozeroy", name="Cumulative £"), row=1, col=1)
fig.add_trace(go.Scatter(x=daily_rev.index, y=daily_rev["cum_avg"],
                         mode="lines", name="Running Avg",
                         line=dict(color="#00CC96", width=2.5)), row=1, col=2)

fig.update_layout(height=380, showlegend=False)
fig.show()

### 8.3 — User-Level Rolling Features (Fraud-Detection Pattern)

This is the **exact pattern** used in Revolut's fraud-scoring pipeline.

In [ ]:
# For each user, compute a 7-day rolling sum of spend
# Step 1: Aggregate to user × day
user_daily = (
    df.groupby(["user_id", "date"])["amount_gbp"]
    .sum()
    .reset_index()
)
user_daily["date"] = pd.to_datetime(user_daily["date"])
user_daily.sort_values(["user_id", "date"], inplace=True)

# Step 2: Apply rolling window per user
user_daily["spend_7d"] = (
    user_daily.groupby("user_id")["amount_gbp"]
    .rolling(7, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

# Step 3: Also compute rolling z-score (anomaly signal)
user_daily["mean_30d"] = (
    user_daily.groupby("user_id")["amount_gbp"]
    .rolling(30, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
user_daily["std_30d"] = (
    user_daily.groupby("user_id")["amount_gbp"]
    .rolling(30, min_periods=1)
    .std()
    .reset_index(level=0, drop=True)
)
user_daily["z_score"] = (user_daily["amount_gbp"] - user_daily["mean_30d"]) / user_daily["std_30d"].replace(0, 1)

print(f"✓ User-day rows: {len(user_daily):,}")
user_daily.head(10)

In [ ]:
# Distribution of z-scores — anomalies are typically |z| > 3
fig = px.histogram(
    user_daily["z_score"].clip(-6, 6),
    nbins=100,
    title="Distribution of User Spending Z-Scores (30-day rolling)",
    labels={"value": "Z-Score"},
    color_discrete_sequence=["#636EFA"],
)
fig.add_vline(-3, line_dash="dash", line_color="red", annotation_text="z = -3")
fig.add_vline(3, line_dash="dash", line_color="red", annotation_text="z = +3")
fig.update_layout(height=380)
fig.show()

n_anomalies = (user_daily["z_score"].abs() > 3).sum()
print(f"Anomalous user-days (|z| > 3): {n_anomalies:,}  "
      f"({n_anomalies / len(user_daily) * 100:.2f}%)")

---
## §9 · `.transform()` for Feature Engineering

`.transform()` broadcasts a group-level statistic **back to every row** in the original DataFrame.
This is essential for creating relative features (e.g., "how does this txn compare to the user's average?").

In [ ]:
# ── Feature 1: Amount as % of user's total spend ─────────────────
user_total = df.groupby("user_id")["amount_gbp"].transform("sum")
df["pct_of_user_total"] = df["amount_gbp"] / user_total * 100

# ── Feature 2: Deviation from user's category-level average ──────
user_cat_mean = df.groupby(["user_id", "merchant_category"])["amount_gbp"].transform("mean")
df["deviation_from_user_cat_mean"] = df["amount_gbp"] - user_cat_mean

# ── Feature 3: User's historical transaction count (at time of txn) ─
df["user_txn_rank"] = df.groupby("user_id").cumcount() + 1

# ── Feature 4: Share of daily volume ─────────────────────────────
daily_total = df.groupby("date")["amount_gbp"].transform("sum")
df["share_of_daily"] = df["amount_gbp"] / daily_total * 100

print("✓ 4 transform-based features engineered")
df[["txn_id", "user_id", "amount_gbp", "pct_of_user_total",
    "deviation_from_user_cat_mean", "user_txn_rank", "share_of_daily"]].head(10)

In [ ]:
# Why .transform() is faster than .apply() for group broadcasts
import time

t0 = time.time()
_ = df.groupby("user_id")["amount_gbp"].transform("mean")
time_transform = time.time() - t0

t0 = time.time()
_ = df.groupby("user_id")["amount_gbp"].apply(lambda x: pd.Series([x.mean()] * len(x), index=x.index))
time_apply = time.time() - t0

print(f".transform('mean'):  {time_transform:.4f} s")
print(f".apply(lambda):      {time_apply:.4f} s")
print(f"Speedup: {time_apply / time_transform:.1f}×")

---
## §10 · Visualization Dashboard

In [ ]:
# ── 10.1 Revenue by Country (choropleth-style bar) ───────────────
fig = px.bar(
    country_summary.reset_index(),
    x="country", y="total_gbp",
    color="revenue_per_user", color_continuous_scale="Viridis",
    title="Total Revenue by Country (color = revenue per user)",
    labels={"total_gbp": "Total Revenue (GBP)", "country": "Country"},
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# ── 10.2 Hourly Transaction Pattern by Channel ───────────────────
hourly_channel = (
    df.groupby(["hour", "channel"])
    .size()
    .unstack(fill_value=0)
)

fig = px.line(
    hourly_channel,
    title="Transaction Volume by Hour and Channel",
    labels={"value": "Transactions", "hour": "Hour of Day", "variable": "Channel"},
)
fig.update_layout(height=400, legend=dict(orientation="h", y=1.12))
fig.show()

In [ ]:
# ── 10.3 Monthly Trend by Category (stacked area) ────────────────
monthly_cat = (
    df.groupby(["year_month", "merchant_category"])["amount_gbp"]
    .sum()
    .unstack(fill_value=0)
)
monthly_cat.index = monthly_cat.index.astype(str)

fig = px.area(
    monthly_cat,
    title="Monthly Revenue by Category (stacked area)",
    labels={"value": "Revenue (GBP)", "year_month": "Month", "variable": "Category"},
)
fig.update_layout(height=440, legend=dict(orientation="h", y=1.12))
fig.show()

In [ ]:
# ── 10.4 Summary Statistics Table ────────────────────────────────
summary = pd.DataFrame({
    "Total Transactions": f"{len(df):,}",
    "Unique Users":       f"{df['user_id'].nunique():,}",
    "Total Revenue (GBP)": f"£{df['amount_gbp'].sum():,.0f}",
    "Avg Transaction":    f"£{df['amount_gbp'].mean():,.2f}",
    "Median Transaction": f"£{df['amount_gbp'].median():,.2f}",
    "Top Category":       df.groupby('merchant_category')['amount_gbp'].sum().idxmax(),
    "Top Channel":        df.groupby('channel')['amount_gbp'].sum().idxmax(),
    "Date Range":         f"{df['timestamp'].min().date()} → {df['timestamp'].max().date()}",
}, index=["Dataset Summary"]).T
summary

---
## §11 · Key Business Takeaway

> ### 🎯 Action Items for a Fintech Data Scientist
>
> | Pattern | Use Case | Key Method |
> |---------|----------|------------|
> | **Basic GroupBy** | Revenue by category/channel | `df.groupby(col).agg(...)` |
> | **Multi-key GroupBy** | Revenue by category × currency | `df.groupby([col1, col2])` |
> | **Named Aggregation** | Multiple stats with clean column names | `.agg(name=(col, func))` |
> | **Custom Functions** | CV, IQR, domain-specific metrics | `.agg(custom_func)` |
> | **Pivot Tables** | Stakeholder-ready cross-tabulations | `pd.pivot_table(...)` |
> | **Rolling Windows** | 7-day spend, fraud velocity signals | `.groupby(user).rolling(7).sum()` |
> | **Expanding Windows** | Cumulative YTD metrics | `.expanding().sum()` |
> | **`.transform()`** | Row-level relative features | `df.groupby(user)[col].transform('mean')` |
> | **`.filter()`** | Keep only high-value user rows | `df.groupby(user).filter(lambda)` |
>
> ### 💡 Pro Tips
>
> 1. **Always use `.transform()`** over `.apply()` when broadcasting group stats back to rows — it's **5–20× faster**.
> 2. **Use named aggregation** (`agg(name=(col, func))`) for self-documenting code that stakeholders can read.
> 3. **Rolling z-scores** (Module §8.3) are the #1 feature in fraud velocity models — they detect when a user's spending suddenly spikes.
> 4. **Pivot tables with margins** (`margins=True`) are the fastest way to produce executive summary tables.
>
> ### 📌 Remember
>
> ```
> .groupby()  → aggregate (reduce rows)
> .transform() → broadcast (keep rows, add context)
> .filter()   → subset (keep groups that match a condition)
> .rolling()  → time-aware aggregation (trend detection)
> ```